# Velociraptor Training Notebook

Train a velociraptor to walk, run, and strike prey using reinforcement learning.

**Training Stages:**
1. **Balance** - Learn to stand without falling
2. **Locomotion** - Walk and run forward
3. **Strike** - Sprint and attack prey with sickle claws

**Supported Algorithms:** PPO, SAC

This notebook loads all configs from the TOML files in `configs/velociraptor/` and supports
the full 3-stage curriculum with either algorithm.

## 1. Setup & Installation

In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system('pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib')
    import subprocess, pathlib
    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system('pip install -q -e /content/mesozoic-labs')
    print("Colab setup complete.")
else:
    print("Running locally.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Choose your algorithm and training parameters here. All reward weights and
hyperparameters are loaded from the TOML configs in `configs/velociraptor/`.

In [ ]:
# ============================================================
# USER CONFIGURATION - Modify these values as needed
# ============================================================

ALGORITHM = "PPO"  # "PPO" or "SAC"
N_ENVS = 4         # Number of parallel environments
SEED = 42          # Random seed for reproducibility
QUICK_TEST = True   # Set to False for full training runs

# ============================================================
# Load configs from TOML files
# ============================================================
from environments.shared.config import load_all_stages

STAGE_CONFIGS = load_all_stages("velociraptor")

algo_key = "ppo_kwargs" if ALGORITHM == "PPO" else "sac_kwargs"

print(f"Algorithm: {ALGORITHM}")
print(f"Parallel envs: {N_ENVS}")
print(f"Quick test: {QUICK_TEST}")
print()
for stage, config in STAGE_CONFIGS.items():
    cur = config.get("curriculum_kwargs", {})
    ts = 50_000 if QUICK_TEST else cur.get("timesteps", 500_000)
    print(f"Stage {stage}: {config['name']} - {config['description']}")
    print(f"  Timesteps: {ts:,}")
    print(f"  {ALGORITHM} hyperparams: {config[algo_key]}")
    print()

## 3. Explore the Environment

In [ ]:
from environments.velociraptor.envs.raptor_env import RaptorEnv

env = RaptorEnv()

print("Environment loaded successfully!")
print(f"\nObservation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"\nAction space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")
print(f"  Range: [{env.action_space.low[0]}, {env.action_space.high[0]}]")

model_mj = env.model
print(f"\nModel Information:")
print(f"  Bodies: {model_mj.nbody}")
print(f"  Joints: {model_mj.njnt}")
print(f"  Actuators: {model_mj.nu}")
print(f"  Sensors: {model_mj.nsensor}")
print(f"  Total DOF: {model_mj.nv}")
print(f"  Total mass: {sum(model_mj.body_mass):.2f} kg")

print("\nActuators:")
for i in range(model_mj.nu):
    name = mujoco.mj_id2name(model_mj, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  [{i:2d}] {name}")

env.close()

In [ ]:
# Run random episodes to establish a baseline
env = RaptorEnv()
n_episodes = 5
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    obs, info = env.reset(seed=ep)
    total_reward = 0
    step = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        if terminated or truncated:
            break
    episode_rewards.append(total_reward)
    episode_lengths.append(step)
    print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

print(f"\nRandom policy baseline:")
print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
env.close()

## 4. Training Infrastructure

In [ ]:
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

ALGO_CLASS = {"PPO": PPO, "SAC": SAC}


def make_env(stage, rank, seed=0):
    """Create a single environment instance."""
    def _init():
        env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
        env = RaptorEnv(**env_kwargs)
        env = Monitor(env)
        env.reset(seed=seed + rank)
        return env
    set_random_seed(seed)
    return _init


def create_vec_env(stage, n_envs=N_ENVS, seed=SEED):
    """Create vectorized environment with observation/reward normalization."""
    env = DummyVecEnv([make_env(stage, i, seed) for i in range(n_envs)])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0)
    return env


def get_algo_kwargs(stage):
    """Get algorithm-specific hyperparameters for a stage."""
    config = STAGE_CONFIGS[stage]
    if ALGORITHM == "PPO":
        return config["ppo_kwargs"].copy()
    else:
        return config["sac_kwargs"].copy()


def train_stage(
    stage,
    timesteps,
    load_path=None,
    log_base=None,
):
    """Train a single curriculum stage. Returns (model, final_model_path)."""
    config = STAGE_CONFIGS[stage]
    AlgoClass = ALGO_CLASS[ALGORITHM]
    algo_kwargs = get_algo_kwargs(stage)
    algo_kwargs["verbose"] = 1

    # Directories
    if log_base is None:
        log_base = Path("../logs")
    stage_dir = log_base / f"velociraptor_{ALGORITHM.lower()}_stage{stage}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    stage_dir.mkdir(parents=True, exist_ok=True)
    model_dir = stage_dir / "models"
    model_dir.mkdir(exist_ok=True)
    algo_kwargs["tensorboard_log"] = str(stage_dir / "tensorboard")

    print(f"{'=' * 60}")
    print(f"Stage {stage}: {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    print(f"{'=' * 60}")

    # Environments
    train_env = create_vec_env(stage)
    eval_env = create_vec_env(stage, n_envs=1, seed=SEED + 1000)

    # Create or load model
    if load_path:
        print(f"Loading model from: {load_path}")
        model = AlgoClass.load(load_path, env=train_env)
        model.learning_rate = algo_kwargs["learning_rate"]
        if ALGORITHM == "PPO":
            model.ent_coef = algo_kwargs["ent_coef"]
            model.clip_range = algo_kwargs["clip_range"]
    else:
        model = AlgoClass("MlpPolicy", train_env, **algo_kwargs)

    # Callbacks
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(model_dir),
        log_path=str(stage_dir),
        eval_freq=max(5000 // N_ENVS, 1),
        n_eval_episodes=5,
        deterministic=True,
    )
    checkpoint_callback = CheckpointCallback(
        save_freq=max(25000 // N_ENVS, 1),
        save_path=str(model_dir),
        name_prefix=f"stage{stage}",
        save_vecnormalize=True,
    )

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=CallbackList([eval_callback, checkpoint_callback]),
        progress_bar=True,
    )

    # Save final model
    final_path = model_dir / f"stage{stage}_final"
    model.save(str(final_path))
    train_env.save(str(final_path) + "_vecnorm.pkl")
    print(f"\nModel saved to: {final_path}.zip")

    # Quick evaluation
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=5)
    print(f"Eval: mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

    train_env.close()
    eval_env.close()

    return model, str(final_path), stage_dir


print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")

## 5. Stage 1: Balance

The raptor learns to stand upright without falling. No forward velocity reward —
just a strong alive bonus.

In [ ]:
cur1 = STAGE_CONFIGS[1].get("curriculum_kwargs", {})
timesteps_1 = 50_000 if QUICK_TEST else cur1.get("timesteps", 500_000)

model_1, path_1, dir_1 = train_stage(stage=1, timesteps=timesteps_1)

## 6. Stage 2: Locomotion

Starting from the Stage 1 checkpoint, the raptor learns to walk and run forward.

In [ ]:
cur2 = STAGE_CONFIGS[2].get("curriculum_kwargs", {})
timesteps_2 = 50_000 if QUICK_TEST else cur2.get("timesteps", 1_000_000)

model_2, path_2, dir_2 = train_stage(stage=2, timesteps=timesteps_2, load_path=path_1)

## 7. Stage 3: Strike

The raptor learns to sprint toward prey and attack with its sickle claws.

In [ ]:
cur3 = STAGE_CONFIGS[3].get("curriculum_kwargs", {})
timesteps_3 = 50_000 if QUICK_TEST else cur3.get("timesteps", 2_000_000)

model_3, path_3, dir_3 = train_stage(stage=3, timesteps=timesteps_3, load_path=path_2)

## 8. Evaluate Final Policy

In [ ]:
def evaluate_trained_policy(model, stage, n_episodes=10):
    """Evaluate a trained policy on the raw (unnormalized) environment."""
    env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
    env = RaptorEnv(**env_kwargs)

    episode_rewards = []
    episode_lengths = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep + 100)
        total_reward = 0
        step = 0
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            step += 1
            if terminated or truncated:
                break
        episode_rewards.append(total_reward)
        episode_lengths.append(step)
        print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

    env.close()
    print(f"\nResults:")
    print(f"  Mean reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
    print(f"  Mean length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
    return episode_rewards, episode_lengths


print(f"Evaluating final Stage 3 policy ({ALGORITHM})...")
rewards_3, lengths_3 = evaluate_trained_policy(model_3, stage=3)

## 9. Training Curves

In [ ]:
def plot_training_curve(stage_dir, stage, algo_name):
    """Plot the evaluation reward curve from a training run."""
    eval_log = Path(stage_dir) / "evaluations.npz"
    if not eval_log.exists():
        print(f"No evaluation log found for stage {stage}.")
        return

    data = np.load(eval_log)
    timesteps = data["timesteps"]
    results = data["results"]
    mean_rewards = np.mean(results, axis=1)
    std_rewards = np.std(results, axis=1)

    plt.plot(timesteps, mean_rewards, label=f"Stage {stage}: {STAGE_CONFIGS[stage]['name']}")
    plt.fill_between(timesteps, mean_rewards - std_rewards, mean_rewards + std_rewards, alpha=0.2)


plt.figure(figsize=(12, 5))
for stage_num, stage_dir in [(1, dir_1), (2, dir_2), (3, dir_3)]:
    plot_training_curve(stage_dir, stage_num, ALGORITHM)

plt.xlabel("Timesteps")
plt.ylabel("Mean Reward")
plt.title(f"Velociraptor {ALGORITHM} - Curriculum Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Record Video

In [ ]:
try:
    import mediapy
    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Install with: pip install mediapy")

if _HAS_MEDIAPY:
    # Record a video of the final trained policy
    RECORD_STAGE = 3
    env_kwargs = STAGE_CONFIGS[RECORD_STAGE]["env_kwargs"].copy()
    render_env = RaptorEnv(render_mode="rgb_array", **env_kwargs)

    obs, _ = render_env.reset(seed=SEED + 2000)
    frames = []
    episode_reward = 0.0

    for _ in range(1000):
        action, _ = model_3.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = render_env.step(action)
        frames.append(render_env.render())
        episode_reward += reward
        if terminated or truncated:
            break

    render_env.close()

    video_path = str(Path(dir_3) / f"velociraptor_{ALGORITHM.lower()}_stage{RECORD_STAGE}.mp4")
    mediapy.write_video(video_path, frames, fps=50)
    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")
    mediapy.show_video(frames, fps=50)

## 11. Cleanup

In [ ]:
print("Training complete!")
print(f"\nAlgorithm: {ALGORITHM}")
print(f"Stage 1 model: {path_1}.zip")
print(f"Stage 2 model: {path_2}.zip")
print(f"Stage 3 model: {path_3}.zip")
print(f"\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")